# Calibration

Sets the behavioural parameters of the base config through small targeted
sweeps. Each section generates a few configs, runs them in parallel, and prints
a recommended value. Recommendations accumulate in `CAL` and feed the later
sections, so run top to bottom, once per geometry.


In [ ]:
import sys, os, json, gzip, re, shutil
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# the notebook may run from notebooks/ or from the repo root
_here = Path('.').resolve()
if (_here / 'configs').exists():
    PROJECT_ROOT = _here
elif (_here.parent / 'configs').exists():
    PROJECT_ROOT = _here.parent
else:
    raise RuntimeError(f"Cannot locate project root from {_here}")

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from generate_configs import ConfigGenerator, CONFIGS_PATH_PREFIX
from batch_run import run_simulation

plt.rcParams['figure.figsize'] = (13, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

CAL = {}
print('Setup complete. PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# 'balanced' (geom 10) or 'imbalanced' (geom 12), run the whole notebook once per variant
GEOM_VARIANT = 'imbalanced'

_GEOM_SETTINGS = {
    'balanced': {'geom': 10, 'base': 'big_city_base_balanced_calibrated.conf',
                 'regions': 'regions_big_city_balanced.json', 'label': 'balanced'},
    'imbalanced': {'geom': 12, 'base': 'big_city_base_imbalanced_calibrated.conf',
                   'regions': 'regions_big_city_extreme_imbalanced.json', 'label': 'imbalanced'}
}
assert GEOM_VARIANT in _GEOM_SETTINGS, f"GEOM_VARIANT must be one of {list(_GEOM_SETTINGS)}"

GEOM = _GEOM_SETTINGS[GEOM_VARIANT]['geom']
BASE_CONF = _GEOM_SETTINGS[GEOM_VARIANT]['base']
REGIONS_FILE = _GEOM_SETTINGS[GEOM_VARIANT]['regions']
print(f'Calibrating for: {GEOM_VARIANT}  (geom={GEOM}, base={BASE_CONF}, regions={REGIONS_FILE})')

In [ ]:
def parse_run_id(run_id):
    p = {}
    m = re.search(r'_d_(\d+(?:\.\d+)?)(?:_|$)', run_id)
    if m: p['d'] = float(m.group(1))
    m = re.search(r'_R_(\d+)_(\d+)(?:_|$)', run_id)
    if m: p['R'] = float(f'{m.group(1)}.{m.group(2)}')
    m = re.search(r'_alg_(.+?)_geom_', run_id)
    if m: p['matching'] = m.group(1)
    m = re.search(r'_geom_(\d+)(?:_|$)', run_id)
    if m: p['geom'] = int(m.group(1))
    m = re.search(r'_behav_(\w+?)(?:_ic|_reset|$)', run_id)
    if m: p['behaviour'] = m.group(1)
    return p


def load_aggregates(results_dir):
    """Load final-snapshot rows from all aggregate files in a directory."""
    rows = []
    for f in sorted(Path(results_dir).glob('run_*_aggregates.csv.gz')):
        try:
            df = pd.read_csv(f, index_col=0)
            if df.empty:
                continue
            run_id = re.sub(r'^run_', '', re.sub(r'_aggregates\.csv\.gz$', '', f.name))
            row = df.iloc[-1].to_dict()
            row['run_id'] = run_id
            row.update(parse_run_id(run_id))
            rows.append(row)
        except Exception as e:
            print(f'  Warning: could not load {f.name}: {e}')
    return pd.DataFrame(rows)


def load_all_batches(results_dir, pattern='run_*_aggregates.csv.gz'):
    """Load every batch row (time series) from all aggregate files."""
    all_dfs = []
    for f in sorted(Path(results_dir).glob(pattern)):
        try:
            df = pd.read_csv(f, index_col=0)
            if df.empty:
                continue
            run_id = re.sub(r'^run_', '', re.sub(r'_aggregates\.csv\.gz$', '', f.name))
            df['run_id'] = run_id
            for k, v in parse_run_id(run_id).items():
                df[k] = v
            all_dfs.append(df)
        except Exception as e:
            print(f'  Warning: {f.name}: {e}')
    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()


def load_safety_scores(results_dir):
    """Return {run_id: {'initial': [...], 'final': [...]}} from per-taxi metrics."""
    out = {}
    for f in sorted(Path(results_dir).glob('run_*_per_taxi_metrics.json.gz')):
        run_id = re.sub(r'^run_', '', re.sub(r'_per_taxi_metrics\.json\.gz$', '', f.name))
        initial = final = None
        with gzip.open(f, 'rt') as fh:
            for line in fh:
                if line.strip():
                    scores = [float(s) for s in json.loads(line).get('safety_score', []) if s is not None]
                    if scores:
                        if initial is None:
                            initial = scores
                        final = scores
        out[run_id] = {'initial': initial or [], 'final': final or []}
    return out


def load_per_taxi_income(results_dir):
    """Return {run_id: [per-taxi final trip_income]} from per-taxi metrics."""
    out = {}
    for f in sorted(Path(results_dir).glob('run_*_per_taxi_metrics.json.gz')):
        run_id = re.sub(r'^run_', '', re.sub(r'_per_taxi_metrics\.json\.gz$', '', f.name))
        final = None
        with gzip.open(f, 'rt') as fh:
            for line in fh:
                if line.strip():
                    data = json.loads(line)
                    income = data.get('trip_income', [])
                    times = data.get('time_serving', [])
                    if income:
                        final = {'income': income, 'time_serving': times, 'trip_num_completed': data.get('trip_num_completed', []),
                                 'time_to_request': data.get('time_to_request', []),
                                 'time_waiting': data.get('time_waiting', []),
                                 'time_cruising': data.get('time_cruising', []),
                                 'time_on_break': data.get('time_on_break', [])}
        if final:
            out[run_id] = final
    return out


def load_service_stats(results_dir):
    """Return {run_id: {'n_done', 'n_dropped', 'service_rate'}} by reading ALL batches."""
    stats = {}
    for f in sorted(Path(results_dir).glob('run_*_per_request_metrics.json.gz')):
        run_id = re.sub(r'^run_', '', re.sub(r'_per_request_metrics\.json\.gz$', '', f.name))
        n_done = n_dropped = 0
        with gzip.open(f, 'rt') as fh:
            for line in fh:
                if line.strip():
                    reqs = json.loads(line).get('requests', [])
                    n_done += sum(1 for r in reqs if r.get('mode') == 'done')
                    n_dropped += sum(1 for r in reqs if r.get('mode') == 'dropped')
        total = n_done + n_dropped
        stats[run_id] = {'n_done': n_done, 'n_dropped': n_dropped,
                         'service_rate': n_done / total if total else None}
    return stats


def run_batch_parallel(config_files, max_workers=None):
    if not config_files:
        print('No configs to run.');
        return [], []
    if max_workers is None:
        max_workers = max(1, (os.cpu_count() or 2) - 1)
    total = len(config_files)
    print(f'Running {total} configs with {max_workers} parallel workers...')
    ok, fail = [], []
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(run_simulation, str(f)): f for f in config_files}
        for i, fut in enumerate(as_completed(futs)):
            success, path, output, _ = fut.result()
            tag = f'[{i + 1}/{total}]'
            if success:
                print(f'  {tag} ok {Path(path).name}')
                ok.append(path)
            else:
                print(f'  {tag} FAIL {Path(path).name}')
                print(f'      {output[:300]}')
                fail.append(path)
    print(f'Done - {len(ok)} OK, {len(fail)} failed')
    return ok, fail


def add_utilization(df):
    active = [c for c in ['avg_time_serving', 'avg_time_waiting',
                          'avg_time_to_request', 'avg_time_cruising'] if c in df.columns]
    if active:
        total = df[active].sum(axis=1)
        df['utilization'] = (df.get('avg_time_serving', 0) / total.where(total > 0)).clip(0, 1)
    return df


def write_configs(configs_and_names, layer_dir):
    Path(layer_dir).mkdir(parents=True, exist_ok=True)
    seen = set()
    written = []
    for fname, content in configs_and_names:
        p = Path(layer_dir) / fname
        if p in seen:
            print(f'  Warning: duplicate config filename skipped: {fname}')
            continue
        seen.add(p)
        p.write_text(content)
        written.append(p)
    print(f'Wrote {len(written)} configs -> {layer_dir}')
    return written


print('Helpers ready')

## Trip economics

No simulation, just a sanity check on the price/cost parameters in the base config:

```
trip_profit = price_fixed + avg_trip_length * price_per_dist
            - avg_trip_length * cost_per_unit
            - avg_trip_time  * cost_per_time
```

This has to come out clearly positive before anything else is worth calibrating.


In [ ]:
with open(f'configs/{BASE_CONF}') as f:
    base = json.load(f)

avg_len = base.get('avg_request_lengths', 50)
velocity = 1
avg_time = avg_len / velocity

revenue = base['price_fixed'] + avg_len * base['price_per_dist']
costs = avg_len * base['cost_per_unit'] + avg_time * base.get('cost_per_time', 0)
profit = revenue - costs
margin = profit / revenue if revenue else 0

print(f'Base config : {BASE_CONF}')
print(f'avg trip    : {avg_len:.1f} units, ~{avg_time:.0f} TU')
print(f'Revenue     : {revenue:.1f}')
print(f'Costs       : {costs:.1f}')
print(f'Profit      : {profit:.1f}  ({margin:.0%} margin)')
if profit > 0:
    print('ok - economics look healthy.')
else:
    print('FAIL - negative profit, adjust price/cost parameters before continuing.')

CAL['base_config'] = BASE_CONF

## Supply / demand balance

Sweeps R at fixed d using `nearest` matching with no breaks and a constant request
rate, so the measurements reflect steady state. Looking for utilization in the
40-65% band with a service rate above 90%.


In [ ]:
SD_BASE = CAL.get('base_config', BASE_CONF)
# characteristic inter-taxi distance, N = area / d^2
SD_D = 258
SD_R_LIST = [0.2, 0.3, 0.4, 0.5, 0.6, 0.8]
SD_GEOM = GEOM
SD_BEHAV = 1
SD_DAYS = 0.25
SD_DIR = f'configs/calibration/layer1_{GEOM_VARIANT}'

In [ ]:
gen1 = ConfigGenerator(SD_BASE, days=SD_DAYS)
configs1 = []
for R in SD_R_LIST:
    conf = gen1.generate_config(SD_D, R, 'nearest', SD_GEOM, SD_BEHAV,
                                no_breaks=True, constant_rate=True)
    if conf is not None:
        fname, content = gen1.dump_config(conf)
        configs1.append((fname, content))

written1 = write_configs(configs1, SD_DIR)
print('Configs:', [p.name for p in written1])

In [ ]:
ok1, fail1 = run_batch_parallel(written1)

In [ ]:
RESULTS_SD = Path(f'results/calibration/layer1_{GEOM_VARIANT}')

df1 = load_aggregates(RESULTS_SD)
if df1.empty:
    print('No results found - run the simulation cells first.')
else:
    df1 = add_utilization(df1)

    svc1 = load_service_stats(RESULTS_SD)
    df1['service_rate'] = df1['run_id'].map({k: v['service_rate'] for k, v in svc1.items()})

    df1_sorted = df1.sort_values('R') if 'R' in df1.columns else df1

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(df1_sorted['R'], df1_sorted['utilization'], 'o-', color='tab:blue')
    axes[0].axhspan(0.40, 0.65, alpha=0.15, color='green', label='target 40-65%')
    axes[0].set_xlabel('R');
    axes[0].set_ylabel('Utilization');
    axes[0].set_title(f'Utilization vs R [{GEOM_VARIANT}]')
    axes[0].legend(fontsize=9)

    if 'service_rate' in df1_sorted.columns:
        axes[1].plot(df1_sorted['R'], df1_sorted['service_rate'], 's-', color='tab:orange')
        axes[1].axhline(0.90, color='green', linestyle='--', label='target > 90%')
        axes[1].set_xlabel('R');
        axes[1].set_ylabel('Service rate');
        axes[1].set_title('Service Rate vs R')
        axes[1].legend(fontsize=9)

    if 'avg_trip_income' in df1_sorted.columns:
        axes[2].plot(df1_sorted['R'], df1_sorted['avg_trip_income'], '^-', color='tab:green')
        axes[2].set_xlabel('R');
        axes[2].set_ylabel('Avg trip income');
        axes[2].set_title('Income vs R')

    plt.tight_layout()
    plt.show()

    show_cols = [c for c in ['R', 'd', 'utilization', 'service_rate', 'avg_trip_income'] if c in df1.columns]
    display(df1_sorted[show_cols].reset_index(drop=True).round(3))

    print('\nRecommendation')
    good = df1[(df1['utilization'] >= 0.40) & (df1['utilization'] <= 0.65)]
    if 'service_rate' in df1.columns:
        good = good[good['service_rate'] >= 0.90]

    if not good.empty:
        good = good.copy()
        good['_score'] = (good['utilization'] - 0.525).abs()
        best = good.loc[good['_score'].idxmin()]
        rec_R = best['R']
        print(f'  ok  R = {rec_R}')
        print(f'    Utilization : {best["utilization"]:.1%}')
        if 'service_rate' in best:
            print(f'    Service rate: {best["service_rate"]:.1%}')
    else:
        df1['_dist'] = (df1['utilization'] - 0.525).abs()
        best = df1.loc[df1['_dist'].idxmin()]
        rec_R = best['R']
        print('  no R meets both criteria.')
        print('    Closest: R = {rec_R}, utilization = {best["utilization"]:.1%}')
        if best['utilization'] > 0.65:
            print('    -> try lower R values')
        elif best['utilization'] < 0.40:
            print('    -> try higher R values')
        if 'service_rate' in best and best['service_rate'] < 0.90:
            print(f'    -> service rate low ({best["service_rate"]:.1%}): raise hard_limit or lower R')

    print(f'  ok  d = {SD_D}')
    CAL['R'] = rec_R
    CAL['d'] = SD_D
    print(f'\n  CAL["R"] = {rec_R}, CAL["d"] = {SD_D}')

## Income baseline

No new runs, reuses the supply/demand results at the chosen R. Measures income per
work-time unit to set `income_target_rate` and per-trip income to set
`satisfaction_income_ref`.


In [ ]:
RESULTS_SD = Path(f'results/calibration/layer1_{GEOM_VARIANT}')

ptm1 = load_per_taxi_income(RESULTS_SD)
if not ptm1:
    print('No per-taxi metrics found. Run the supply/demand sweep first.')
else:
    chosen_R = CAL.get('R')
    target_runs = {k: v for k, v in ptm1.items() if f'_R_{str(chosen_R).replace(".", "_")}' in k
                   or (chosen_R is None)}
    if not target_runs:
        target_runs = ptm1
        print(f'  Note: using all supply/demand runs (could not filter by R={chosen_R})')

    all_income_rates = []
    all_trip_incomes = []
    for run_id, data in target_runs.items():
        incomes = [float(x) for x in data['income'] if x is not None]
        t_serve = [float(x) for x in data['time_serving'] if x is not None]
        t_req = [float(x) for x in data.get('time_to_request', [0] * len(incomes)) if x is not None]
        t_wait = [float(x) for x in data.get('time_waiting', [0] * len(incomes)) if x is not None]
        t_cruise = [float(x) for x in data.get('time_cruising', [0] * len(incomes)) if x is not None]
        for i in range(min(len(incomes), len(t_serve))):
            work_time = t_serve[i] + (t_req[i] if i < len(t_req) else 0) + \
                        (t_wait[i] if i < len(t_wait) else 0) + (t_cruise[i] if i < len(t_cruise) else 0)
            if work_time > 1:
                all_income_rates.append(incomes[i] / work_time)
        # satisfaction_income_ref normalizes a SINGLE trip in city_model (tanh),
        # so calibrate on per-trip income, not per-taxi cumulative income.
        counts = data.get('trip_num_completed', [])
        raw_inc = data['income']
        for _i in range(min(len(raw_inc), len(counts))):
            _inc, _cnt = raw_inc[_i], counts[_i]
            if _inc is not None and _cnt:
                all_trip_incomes.append(float(_inc) / float(_cnt))

    if not all_income_rates:
        print('Could not compute income rates - check per-taxi metrics.')
    else:
        mean_rate = float(np.mean(all_income_rates))
        p80_rate = float(np.percentile(all_income_rates, 80))
        median_income = float(np.median(all_trip_incomes)) if all_trip_incomes else None
        p70_income = float(np.percentile(all_trip_incomes, 70)) if all_trip_incomes else None

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(all_income_rates, bins=30, color='tab:blue', edgecolor='white')
        axes[0].axvline(mean_rate, color='red', linestyle='--', label=f'mean {mean_rate:.4f}')
        axes[0].axvline(p80_rate, color='orange', linestyle='--', label=f'80th pct {p80_rate:.4f}')
        axes[0].set_xlabel('Income / work-time unit');
        axes[0].set_title('Income Rate Distribution')
        axes[0].legend(fontsize=9)

        if all_trip_incomes:
            axes[1].hist(all_trip_incomes, bins=30, color='tab:green', edgecolor='white')
            axes[1].axvline(median_income, color='red', linestyle='--', label=f'median {median_income:.1f}')
            axes[1].axvline(p70_income, color='orange', linestyle='--', label=f'70th pct {p70_income:.1f}')
            axes[1].set_xlabel('Trip income');
            axes[1].set_title('Trip Income Distribution')
            axes[1].legend(fontsize=9)

        plt.tight_layout();
        plt.show()

        rec_itr = round(mean_rate * 0.85, 6)
        rec_iref = round(p70_income, 1) if p70_income else None

        print('Recommendation')
        print(f'  Mean income rate          : {mean_rate:.4f} / TU')
        print(f'  income_target_rate        : {rec_itr:.4f}  (85% of mean)')
        print(f'  80% of mean               : {mean_rate * 0.80:.4f}  (stricter)')
        if rec_iref:
            print(f'  satisfaction_income_ref   : {rec_iref:.1f}  (70th pct trip income)')
        print()
        print('  Add to base config:')
        print(f'    "income_target_rate": {rec_itr:.6f}')
        if rec_iref:
            print(f'    "satisfaction_income_ref": {rec_iref:.1f}')

        CAL['income_target_rate'] = rec_itr
        CAL['mean_income_rate'] = mean_rate
        CAL['satisfaction_income_ref'] = rec_iref
        CAL['median_trip_income'] = median_income
        print(f'\n  CAL["income_target_rate"] = {rec_itr}')

## Safety score decay

Sweeps `safety_score_change_serving_rate` so that the safety score drops about
20-40 points over a long shift.


In [ ]:
SAF_BASE = CAL.get('base_config', BASE_CONF)
SAF_D = CAL.get('d', 258)
SAF_R = CAL.get('R', 0.5)
SAF_GEOM = GEOM
SAF_BEHAV = 1
# 0.25 days is roughly one long shift (2200 TU with big_city_base)
SAF_DAYS = 0.25
SAF_DIR = f'configs/calibration/layer3_{GEOM_VARIANT}'

# negative: degradation per TU while serving
SAF_SERVING_RATES = [-0.005, -0.010, -0.015, -0.020, -0.025, -0.030]

# recovery constant C in R(t) = t / (t + C)
SAF_RECOVERY_C = 180.0

In [ ]:
gen3 = ConfigGenerator(SAF_BASE, days=SAF_DAYS)
configs3 = []
for rate in SAF_SERVING_RATES:
    conf = gen3.generate_config(SAF_D, SAF_R, 'nearest', SAF_GEOM, SAF_BEHAV,
                                no_breaks=True, constant_rate=True)
    if conf is None:
        continue
    conf['safety_score_change_serving_rate'] = rate
    conf['safety_score_break_recovery_constant'] = SAF_RECOVERY_C
    rate_tag = str(abs(rate)).replace('.', 'p')
    fname, content = gen3.dump_config(conf)
    fname = fname.replace('.conf', f'_srate_{rate_tag}.conf')
    conf_dict = json.loads(content)
    conf_dict['safety_score_change_serving_rate'] = rate
    conf_dict['_srate_tag'] = rate
    content = json.dumps(conf_dict, indent=4) + '\n'
    configs3.append((fname, content))

written3 = write_configs(configs3, SAF_DIR)
print('Configs:', [p.name for p in written3])

In [ ]:
ok3, fail3 = run_batch_parallel(written3)

In [ ]:
RESULTS_SAF = Path(f'results/calibration/layer3_{GEOM_VARIANT}')

safety3 = load_safety_scores(RESULTS_SAF)
if not safety3:
    print('No safety data found - run the sweep first.')
else:
    rows3 = []
    for run_id, sc in safety3.items():
        if not sc['initial'] or not sc['final']:
            continue
        mean_drop = np.mean(sc['initial']) - np.mean(sc['final'])
        m = re.search(r'_srate_(\d+p\d+)', run_id)
        srate = -float(m.group(1).replace('p', '.')) if m else None
        rows3.append({'run_id': run_id, 'srate': srate,
                      'mean_initial': np.mean(sc['initial']),
                      'mean_final': np.mean(sc['final']),
                      'mean_drop': mean_drop})

    df3 = pd.DataFrame(rows3).sort_values('srate')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(df3['srate'].abs(), df3['mean_drop'], 'o-', color='tab:red')
    axes[0].axhspan(20, 40, alpha=0.15, color='green', label='target 20-40 pt')
    axes[0].set_xlabel('|safety_score_change_serving_rate|')
    axes[0].set_ylabel('Mean safety score drop')
    axes[0].set_title('Safety Drop vs Serving Rate')
    axes[0].legend(fontsize=9)

    axes[1].scatter(df3['srate'].abs(), df3['mean_initial'], label='initial', marker='o')
    axes[1].scatter(df3['srate'].abs(), df3['mean_final'], label='final', marker='s')
    axes[1].set_xlabel('|serving rate|')
    axes[1].set_title('Initial vs Final Safety Score')
    axes[1].legend(fontsize=9)

    plt.tight_layout();
    plt.show()
    display(df3[['srate', 'mean_initial', 'mean_final', 'mean_drop']].reset_index(drop=True).round(3))

    in_range = df3[(df3['mean_drop'] >= 20) & (df3['mean_drop'] <= 40)]
    print('\nRecommendation')
    if not in_range.empty:
        best3 = in_range.iloc[len(in_range) // 2]
        rec_rate = best3['srate']
        print(f'  ok  safety_score_change_serving_rate = {rec_rate}')
        print(f'    Mean drop: {best3["mean_drop"]:.1f} points')
    else:
        target_drop = 30
        df3s = df3.dropna(subset=['srate']).sort_values('srate')
        if len(df3s) >= 2:
            rec_rate = float(np.interp(target_drop, df3s['mean_drop'], df3s['srate']))
            print(f'  ~ interpolated serving_rate ~= {rec_rate:.4f} (targets 30 pt drop)')
        else:
            rec_rate = SAF_SERVING_RATES[len(SAF_SERVING_RATES) // 2]
            print(f'  ? not enough data, defaulting to {rec_rate}')

    long_shift_tu = 2880
    expected_drop = long_shift_tu * 0.60 * abs(rec_rate)
    recovery_1440 = 1440 / (1440 + SAF_RECOVERY_C)
    print(f'\n  Recovery constant C = {SAF_RECOVERY_C}')
    print(f'  1440-TU rest recovers {recovery_1440:.0%} of the drop')
    print(f'  Expected drop at 60% utilization over long shift: {expected_drop:.1f} pts')
    if recovery_1440 < 0.75:
        print('  -> recovery slow; consider reducing C')
    elif recovery_1440 > 0.95:
        print('  -> recovery very fast; consider increasing C')

    print(f'\n  Add to base config:')
    print(f'    "safety_score_change_serving_rate": {rec_rate}')
    print(f'    "safety_score_break_recovery_constant": {SAF_RECOVERY_C}')

    CAL['safety_score_change_serving_rate'] = rec_rate
    CAL['safety_score_break_recovery_constant'] = SAF_RECOVERY_C
    print(f'\n  CAL["safety_score_change_serving_rate"] = {rec_rate}')

### Break recovery

Confirms `safety_score_break_recovery_constant` by simulation, since the value
above was picked analytically. Full day runs with breaks enabled, sweeping the
constant C. A typical break should recover 50-80% of the drop accumulated since
the previous break.


In [ ]:
REC_BASE = CAL.get('base_config', BASE_CONF)
REC_D = CAL.get('d', 258)
REC_R = CAL.get('R', 0.5)
REC_GEOM = GEOM
REC_BEHAV = 1
# full day so multiple break cycles occur
REC_DAYS = 1
REC_DIR = f'configs/calibration/layer3b_{GEOM_VARIANT}'
REC_RECOVERY_CONSTANTS = [60, 120, 180, 240, 360]
REC_SERVING_RATE = CAL.get('safety_score_change_serving_rate', -0.025)

gen3b = ConfigGenerator(REC_BASE, days=REC_DAYS)
configs3b = []
for C in REC_RECOVERY_CONSTANTS:
    conf = gen3b.generate_config(REC_D, REC_R, 'nearest', REC_GEOM, REC_BEHAV,
                                 no_breaks=False, constant_rate=True)
    if conf is None:
        continue
    conf['safety_score_change_serving_rate'] = REC_SERVING_RATE
    conf['safety_score_break_recovery_constant'] = float(C)
    fname, content = gen3b.dump_config(conf)
    srate_tag = str(abs(REC_SERVING_RATE)).replace('.', 'p')
    fname = fname.replace('.conf', f'_srate_{srate_tag}_recov_{C}.conf')
    conf_dict = json.loads(content)
    conf_dict['safety_score_break_recovery_constant'] = float(C)
    conf_dict['_recov_tag'] = C
    configs3b.append((fname, json.dumps(conf_dict, indent=4) + '\n'))

written3b = write_configs(configs3b, REC_DIR)
print('Configs:', [p.name for p in written3b])

In [ ]:
ok3b, fail3b = run_batch_parallel(written3b)

In [ ]:
RESULTS_REC = Path(f'results/calibration/layer3b_{GEOM_VARIANT}')


def load_break_recovery(results_dir):
    """Measure mean safety score recovery during break periods from per-taxi time series.

    pre_drop is accumulated over the FULL serving period since the last break end
    (not just one batch step), so recovery_ratio = recovery / pre_drop is directly
    comparable to the 50-80% target band.
    """
    out = {}
    for f in sorted(Path(results_dir).glob('run_*_per_taxi_metrics.json.gz')):
        run_id = re.sub(r'^run_', '', re.sub(r'_per_taxi_metrics\.json\.gz$', '', f.name))
        batches = []
        with gzip.open(f, 'rt') as fh:
            for line in fh:
                if line.strip():
                    batches.append(json.loads(line))
        if len(batches) < 4:
            continue

        n_taxis = len(batches[0].get('safety_score', []))
        recovery_deltas, pre_drop_deltas = [], []

        for taxi_idx in range(n_taxis):
            scores = [b['safety_score'][taxi_idx] for b in batches
                      if taxi_idx < len(b.get('safety_score', []))]
            breaks = [bool(b.get('on_break', [False] * n_taxis)[taxi_idx]) for b in batches
                      if taxi_idx < len(b.get('on_break', []))]
            if len(scores) < len(batches):
                continue

            serving_start_score = None
            i = 0
            while i < len(breaks) - 1:
                if not breaks[i]:
                    # start of a serving period (first non-break step after a break)
                    if serving_start_score is None or (i > 0 and breaks[i - 1]):
                        serving_start_score = scores[i]

                    if breaks[i + 1]:
                        # last serving step before a break: measure the accumulated drop
                        j = i + 1
                        while j < len(breaks) and breaks[j]:
                            j += 1
                        if j < len(scores) and serving_start_score is not None:
                            recovery = scores[j] - scores[i]
                            pre_drop = serving_start_score - scores[i]
                            if pre_drop > 0:
                                recovery_deltas.append(recovery)
                                pre_drop_deltas.append(pre_drop)
                        serving_start_score = None
                        i = j
                    else:
                        i += 1
                else:
                    serving_start_score = None
                    i += 1

        m = re.search(r'_recov_(\d+)', run_id)
        C = int(m.group(1)) if m else None
        mean_rec = float(np.mean(recovery_deltas)) if recovery_deltas else 0.0
        mean_drop = float(np.mean(pre_drop_deltas)) if pre_drop_deltas else 1.0
        out[run_id] = {
            'C': C,
            'break_events': len(recovery_deltas),
            'mean_recovery': mean_rec,
            'mean_pre_drop': mean_drop,
            'recovery_ratio': mean_rec / mean_drop if mean_drop > 0 else 0.0,
        }
    return out


rec3b = load_break_recovery(RESULTS_REC)
if not rec3b:
    print('No results - run the break recovery sweep first.')
else:
    rows3b = sorted([v for v in rec3b.values() if v['C'] is not None], key=lambda x: x['C'])
    df3b = pd.DataFrame(rows3b)
    display(df3b[['C', 'break_events', 'mean_recovery', 'mean_pre_drop', 'recovery_ratio']].round(3))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df3b['C'], df3b['recovery_ratio'], 'o-', color='tab:green')
    ax.axhspan(0.50, 0.80, alpha=0.15, color='green', label='target 50-80% recovery ratio')
    ax.set_xlabel('safety_score_break_recovery_constant C')
    ax.set_ylabel('Recovery ratio per break event\n(recovery / full serving-period drop)')
    ax.set_title('Break Recovery vs Constant C')
    ax.legend(fontsize=9)
    plt.tight_layout();
    plt.show()

    in_range3b = df3b[(df3b['recovery_ratio'] >= 0.50) & (df3b['recovery_ratio'] <= 0.80)]
    print('\nRecommendation')
    if not in_range3b.empty:
        best3b = in_range3b.iloc[len(in_range3b) // 2]
        rec_C = int(best3b['C'])
        print(f'  ok  safety_score_break_recovery_constant = {rec_C}')
        print(f'    Recovery ratio: {best3b["recovery_ratio"]:.1%} per break event')
    elif not df3b.empty:
        df3b_v = df3b.dropna(subset=['C']).copy()
        df3b_v['dist'] = (df3b_v['recovery_ratio'] - 0.65).abs()
        best3b = df3b_v.loc[df3b_v['dist'].idxmin()]
        rec_C = int(best3b['C'])
        print(f'  ~ closest to 65% target: C = {rec_C}  (recovery ratio: {best3b["recovery_ratio"]:.1%})')
        print(f'    Consider extending REC_RECOVERY_CONSTANTS range.')
    else:
        rec_C = int(CAL.get('safety_score_break_recovery_constant', 180))
        print(f'  ? no break events detected; keeping C = {rec_C}')

    prev_C = int(CAL.get('safety_score_break_recovery_constant', 180))
    if abs(rec_C - prev_C) > 30:
        print(f'\n  ** C updated from {prev_C} (analytical) to {rec_C} (simulation-confirmed)')
        CAL['safety_score_break_recovery_constant'] = rec_C
    else:
        print(f'\n  Analytical C = {prev_C} confirmed by simulation (suggested {rec_C})')
    print(f'  CAL["safety_score_break_recovery_constant"] = {CAL["safety_score_break_recovery_constant"]}')

## Satisfaction normalization

No new runs. Sets `satisfaction_income_ref` and `satisfaction_change_waiting_rate`
so the income and waiting components of satisfaction sit on comparable scales.


In [ ]:
RESULTS_SD = Path(f'results/calibration/layer1_{GEOM_VARIANT}')

R_chosen = CAL.get('R', 0.5)
with open(f'configs/{CAL.get("base_config", BASE_CONF)}') as f:
    base_conf_sat = json.load(f)

avg_len_sat = base_conf_sat.get('avg_request_lengths', 50)
velocity_sat = 1

inter_trip_tu = avg_len_sat / (velocity_sat * R_chosen)

income_ref = CAL.get('satisfaction_income_ref') or CAL.get('median_trip_income')

typical_sat_per_trip = 0.3
if inter_trip_tu > 0:
    rec_wait_rate = -(typical_sat_per_trip / inter_trip_tu)
else:
    rec_wait_rate = -0.003

print('Recommendation')
print(f'  Avg request length       : {avg_len_sat} units')
print(f'  Chosen R                 : {R_chosen}')
print(f'  Est. inter-trip interval : {inter_trip_tu:.0f} TU')
print()
if income_ref:
    print(f'  satisfaction_income_ref  : {income_ref:.1f}')
else:
    income_ref = 50
    print(f'  income_ref not set from the income baseline; using fallback = {income_ref}')

print(f'  satisfaction_change_waiting_rate : {rec_wait_rate:.5f} / TU')
print(f'    ({inter_trip_tu:.0f} TU idle costs {abs(rec_wait_rate * inter_trip_tu):.2f} sat = one typical trip)')
print()
print('  Add to base config:')
print(f'    "satisfaction_income_ref": {income_ref:.1f}')
print(f'    "satisfaction_change_waiting_rate": {rec_wait_rate:.5f}')
print(f'    "satisfaction_income_weight": 0.5')

CAL['satisfaction_income_ref'] = income_ref
CAL['satisfaction_change_waiting_rate'] = rec_wait_rate
print(f'\n  CAL["satisfaction_income_ref"] = {income_ref:.1f}')
print(f'  CAL["satisfaction_change_waiting_rate"] = {rec_wait_rate:.5f}')

## Driver flexibility

Places `driver_flexibility_threshold` in the upper tail (p85) of the income
shortfall distribution and picks the sigmoid temperature. Needs
`income_target_rate` from the income baseline.


In [ ]:
FLEX_BASE = CAL.get('base_config', BASE_CONF)
FLEX_D = CAL.get('d', 258)
FLEX_R = CAL.get('R', 0.5)
# at R=0.5 almost nobody earns below target, so the extra low-demand runs
# are where the flexibility mechanism actually activates
FLEX_R_LOW = 0.3
FLEX_GEOM = GEOM
FLEX_BEHAV = 1
FLEX_DAYS = 1
FLEX_DIR = f'configs/calibration/layer5_{GEOM_VARIANT}'

# lower T = sharper switch at the threshold, higher T = more gradual ramp
FLEX_TEMPERATURES = [0.05, 0.10, 0.15, 0.20, 0.30]

In [ ]:
gen5 = ConfigGenerator(FLEX_BASE, days=FLEX_DAYS)

income_target = CAL.get('income_target_rate')
if income_target is None:
    print('WARNING: income_target_rate not set in CAL - run the income baseline first.')
    income_target = 0.001

configs5 = []
for R_test in [FLEX_R, FLEX_R_LOW]:
    for T in FLEX_TEMPERATURES:
        conf = gen5.generate_config(FLEX_D, R_test, 'nearest_distance_pref', FLEX_GEOM, FLEX_BEHAV,
                                    no_breaks=False, constant_rate=False)
        if conf is None:
            continue
        conf['income_target_rate'] = income_target
        # placeholder, the real threshold comes out of the shortfall analysis
        conf['driver_flexibility_threshold'] = 0.5
        conf['driver_flexibility_temperature'] = T
        fname, content = gen5.dump_config(conf)
        T_tag = str(T).replace('.', 'p')
        fname = fname.replace('.conf', f'_ftemp_{T_tag}.conf')
        conf_dict = json.loads(content)
        conf_dict['_ftemp_tag'] = T
        configs5.append((fname, json.dumps(conf_dict, indent=4) + '\n'))

written5 = write_configs(configs5, FLEX_DIR)
print('Configs:', [p.name for p in written5])

In [ ]:
ok5, fail5 = run_batch_parallel(written5)

In [ ]:
RESULTS_FLEX = Path(f'results/calibration/layer5_{GEOM_VARIANT}')

ptm5 = load_per_taxi_income(RESULTS_FLEX)

if not ptm5:
    print('No shortfall data - run the sweep first.')
else:
    # shortfall is per clock-time (work + break): income per work-TU is near-constant
    # across R, so only the clock rate differentiates drivers
    run_data = []
    for run_id, data in ptm5.items():
        m_T = re.search(r'_ftemp_([\dp]+)', run_id)
        T = float(m_T.group(1).replace('p', '.')) if m_T else None
        R_run = parse_run_id(run_id).get('R', FLEX_R)

        incomes = [float(x) for x in data['income'] if x is not None]
        t_serve = [float(x) for x in data['time_serving'] if x is not None]
        t_req = [float(x) for x in data.get('time_to_request', []) if x is not None]
        t_wait = [float(x) for x in data.get('time_waiting', []) if x is not None]
        t_cruise = [float(x) for x in data.get('time_cruising', []) if x is not None]
        t_break = [float(x) for x in data.get('time_on_break', []) if x is not None]

        clock_rates = []
        for i in range(min(len(incomes), len(t_serve))):
            work = (t_serve[i]
                    + (t_req[i] if i < len(t_req) else 0)
                    + (t_wait[i] if i < len(t_wait) else 0)
                    + (t_cruise[i] if i < len(t_cruise) else 0))
            brk = t_break[i] if i < len(t_break) else 0
            clock = work + brk
            if clock > 1:
                clock_rates.append(incomes[i] / clock)

        if clock_rates:
            run_data.append({'run_id': run_id, 'temperature': T, 'R': R_run,
                             'clock_rates': clock_rates})

    if not run_data:
        print('Could not compute clock-time income rates - check per-taxi metrics.')
    else:
        primary = [r for r in run_data if abs(r['R'] - FLEX_R) < 0.01]
        stress = [r for r in run_data if abs(r['R'] - FLEX_R_LOW) < 0.01]

        all_primary_rates = [rate for r in primary for rate in r['clock_rates']]
        income_target_clock = float(np.mean(all_primary_rates)) if all_primary_rates else None

        if income_target_clock is None:
            print('No primary runs found - check FLEX_R and results.')
        else:
            print(f'  income target (mean clock-rate at R={FLEX_R}): {income_target_clock:.4f} / clock-TU')


            def compute_shortfalls(clock_rates, target):
                return [max(0.0, target - r) / target for r in clock_rates]


            for r in run_data:
                r['shortfalls'] = compute_shortfalls(r['clock_rates'], income_target_clock)

            primary_all_sf = [s for r in primary for s in r['shortfalls']]
            # p85 so that roughly 15% of drivers trigger flexibility at any time
            rec_threshold = float(np.percentile(primary_all_sf, 85)) if primary_all_sf else 0.0
            n_above = sum(s > rec_threshold for s in primary_all_sf)
            pct_above = 100 * n_above / max(len(primary_all_sf), 1)

            print(f'  driver_flexibility_threshold = {rec_threshold:.4f}  (p85 of shortfall at R={FLEX_R})')
            print(f'  Drivers above threshold: {pct_above:.0f}% (target ~15%)')


            def mean_flex(shortfalls, threshold, T):
                return float(np.mean([1 / (1 + np.exp(-(s - threshold) / max(T, 1e-9)))
                                      for s in shortfalls]))


            rows5 = []
            for T_val in sorted(set(r['temperature'] for r in primary if r['temperature'] is not None)):
                runs_T = [r for r in primary if r['temperature'] == T_val]
                all_sf = [s for r in runs_T for s in r['shortfalls']]
                if all_sf:
                    rows5.append({'temperature': T_val, 'R': FLEX_R,
                                  'p85': float(np.percentile(all_sf, 85)),
                                  'mean_flex': mean_flex(all_sf, rec_threshold, T_val)})
            df5r = pd.DataFrame(rows5).sort_values('temperature') if rows5 else pd.DataFrame()

            fig, axes = plt.subplots(1, 3, figsize=(15, 4))

            for r in primary:
                if r['temperature'] is not None:
                    axes[0].hist(r['shortfalls'], bins=30, alpha=0.4, label=f'T={r["temperature"]}')
            axes[0].axvline(rec_threshold, color='red', linestyle='--',
                            label=f'threshold={rec_threshold:.3f}')
            axes[0].set_title(f'Shortfall distribution (R={FLEX_R}, per clock-TU)')
            axes[0].set_xlabel('Shortfall fraction  (1 - income/mean_target)')
            axes[0].legend(fontsize=8)

            if not df5r.empty:
                axes[1].plot(df5r['temperature'], df5r['mean_flex'], 'o-', color='tab:blue')
                axes[1].axhspan(0.10, 0.20, alpha=0.15, color='green', label='target 0.10-0.20')
                axes[1].axhline(0.15, color='green', linestyle='--', alpha=0.5)
                axes[1].set_title('Mean flexibility vs temperature')
                axes[1].set_xlabel('driver_flexibility_temperature')
                axes[1].set_ylabel('mean_flex  (fraction of flexible drivers)')
                axes[1].legend(fontsize=9)

            all_R_vals = sorted(set(r['R'] for r in run_data))
            p85_by_R = []
            for R_val in all_R_vals:
                sf_R = [s for r in run_data if r['R'] == R_val for s in r['shortfalls']]
                if sf_R:
                    p85_by_R.append({'R': R_val, 'p85': np.percentile(sf_R, 85)})
            if p85_by_R:
                df_r = pd.DataFrame(p85_by_R)
                axes[2].plot(df_r['R'], df_r['p85'], 'o-', color='tab:orange')
                axes[2].axhline(rec_threshold, color='red', linestyle='--',
                                label=f'chosen threshold={rec_threshold:.3f}')
                axes[2].set_title('p85 shortfall vs demand level R\n(higher p85 = more drivers struggle)')
                axes[2].set_xlabel('R')
                axes[2].set_ylabel('p85 shortfall')
                axes[2].legend(fontsize=9)

            plt.tight_layout();
            plt.show()
            display(df5r[['temperature', 'R', 'p85', 'mean_flex']].round(4) if not df5r.empty else pd.DataFrame())

            print('\nRecommendation')
            if not df5r.empty:
                in_range5 = df5r[(df5r['mean_flex'] >= 0.10) & (df5r['mean_flex'] <= 0.20)]
                if not in_range5.empty:
                    rec_T5 = float(in_range5.iloc[0]['temperature'])
                    print(f'  ok  driver_flexibility_threshold   = {rec_threshold:.4f}  (p85 shortfall at R={FLEX_R})')
                    print(f'  ok  driver_flexibility_temperature = {rec_T5}')
                    print(f'    Mean flex at T={rec_T5}: {in_range5.iloc[0]["mean_flex"]:.3f}')
                else:
                    df5r_v = df5r.copy()
                    df5r_v['dist'] = (df5r_v['mean_flex'] - 0.15).abs()
                    best5 = df5r_v.loc[df5r_v['dist'].idxmin()]
                    rec_T5 = float(best5['temperature'])
                    print(f'  ~  driver_flexibility_threshold   = {rec_threshold:.4f}  (p85 shortfall at R={FLEX_R})')
                    print(f'  ~  driver_flexibility_temperature = {rec_T5}  (closest to target)')
                    print(f'    Mean flex at T={rec_T5}: {best5["mean_flex"]:.3f}  (target: 0.10-0.20)')
            else:
                rec_T5 = 0.10
                print(f'  ? no temperature sweep data; defaulting to T={rec_T5}')

            if stress:
                stress_all_sf = [s for r in stress for s in r['shortfalls']]
                stress_p85 = np.percentile(stress_all_sf, 85) if stress_all_sf else 0
                print(f'\n  Stress check at R={FLEX_R_LOW}: p85 shortfall = {stress_p85:.4f}')
                if stress_p85 > rec_threshold:
                    print(f'  -> mechanism more active at low demand (expected)')
                else:
                    print(f'  -> mechanism similar across demand levels')

            print()
            print('  Add to base config:')
            print(f'    "driver_flexibility_threshold": {rec_threshold:.4f}')
            print(f'    "driver_flexibility_temperature": {rec_T5}')

            CAL['driver_flexibility_threshold'] = rec_threshold
            CAL['driver_flexibility_temperature'] = rec_T5
            print(f'\n  CAL["driver_flexibility_threshold"] = {rec_threshold:.4f}')
            print(f'  CAL["driver_flexibility_temperature"] = {rec_T5}')


## Route length preference strength

Checks that the preference strength range produces a clear gap in accepted
route lengths between short and long preference drivers. These are design knobs
rather than empirical values, so the sweep only has to make the gap visible.
Applies to `nearest_distance_pref` only. The region preference section below
covers the regional analogue.


In [ ]:
ROUTE_BASE = CAL.get('base_config', BASE_CONF)
ROUTE_D = CAL.get('d', 258)
ROUTE_R = CAL.get('R', 0.5)
ROUTE_GEOM = GEOM
ROUTE_BEHAV = 1
ROUTE_DAYS = 1
ROUTE_DIR = f'configs/calibration/layer6_{GEOM_VARIANT}'

ROUTE_PREF_HIGH_LIST = [0.5, 0.7, 0.9]
ROUTE_PREF_LOW = 0.2
ROUTE_NONPREF_CEILING = 0.25
ROUTE_BASE_ACCEPT_PROB = 0.90

In [ ]:
gen6 = ConfigGenerator(ROUTE_BASE, days=ROUTE_DAYS)
configs6 = []
for high in ROUTE_PREF_HIGH_LIST:
    conf = gen6.generate_config(ROUTE_D, ROUTE_R, 'nearest_distance_pref', ROUTE_GEOM, ROUTE_BEHAV,
                                no_breaks=False, constant_rate=False)
    if conf is None:
        continue
    conf['route_pref_strength_range'] = {'low': ROUTE_PREF_LOW, 'high': high}
    conf['nonpreferred_accept_ceiling'] = ROUTE_NONPREF_CEILING
    conf['preference_base_acceptance_prob'] = ROUTE_BASE_ACCEPT_PROB
    if CAL.get('income_target_rate'):
        conf['income_target_rate'] = CAL['income_target_rate']
    if CAL.get('driver_flexibility_threshold'):
        conf['driver_flexibility_threshold'] = CAL['driver_flexibility_threshold']
        conf['driver_flexibility_temperature'] = CAL.get('driver_flexibility_temperature', 0.15)
    fname, content = gen6.dump_config(conf)
    high_tag = str(high).replace('.', 'p')
    fname = fname.replace('.conf', f'_prefhigh_{high_tag}.conf')
    conf_dict = json.loads(content)
    conf_dict['_prefhigh_tag'] = high
    configs6.append((fname, json.dumps(conf_dict, indent=4) + '\n'))

written6 = write_configs(configs6, ROUTE_DIR)
print('Configs:', [p.name for p in written6])

In [ ]:
ok6, fail6 = run_batch_parallel(written6)

In [ ]:
RESULTS_ROUTE = Path(f'results/calibration/layer6_{GEOM_VARIANT}')


def load_route_lengths_by_pref(results_dir):
    """Join per-request metrics with taxi_static to get driver route preference per trip.
    The route preference is a taxi attribute stored in taxi_static, not in per-request data."""
    out = {}
    result_path = Path(results_dir)
    for req_f in sorted(result_path.glob('run_*_per_request_metrics.json.gz')):
        run_id = re.sub(r'^run_', '', re.sub(r'_per_request_metrics\.json\.gz$', '', req_f.name))

        static_f = result_path / f'run_{run_id}_taxi_static.json.gz'
        taxi_pref = {}
        if static_f.exists():
            with gzip.open(static_f, 'rt') as fh:
                for line in fh:
                    if line.strip():
                        s = json.loads(line)
                        taxi_pref = dict(zip(s.get('taxi_ids', []),
                                             s.get('route_length_pref', [])))
                        break

        short_lens, long_lens, neutral_lens = [], [], []
        with gzip.open(req_f, 'rt') as fh:
            for line in fh:
                if line.strip():
                    for req in json.loads(line).get('requests', []):
                        if req.get('mode') != 'done':
                            continue
                        o = req.get('origin', [0, 0])
                        d = req.get('destination', [0, 0])
                        length = abs(o[0] - d[0]) + abs(o[1] - d[1])
                        if length == 0:
                            pu, do = req.get('pickup'), req.get('dropoff')
                            if pu is not None and do is not None:
                                length = abs(do - pu)
                        pref = taxi_pref.get(req.get('taxi_id'))
                        if pref == 'short_pref':
                            short_lens.append(float(length))
                        elif pref == 'long_pref':
                            long_lens.append(float(length))
                        elif pref == 'neutral_pref':
                            neutral_lens.append(float(length))
        out[run_id] = {'short': short_lens, 'long': long_lens, 'neutral': neutral_lens}
    return out


route_data6 = load_route_lengths_by_pref(RESULTS_ROUTE)
df6 = load_aggregates(RESULTS_ROUTE)

if not route_data6:
    print('No route data found - run the sweep first.')
else:
    rows6 = []
    for run_id, rd in route_data6.items():
        m = re.search(r'_prefhigh_([\dp]+)', run_id)
        high = float(m.group(1).replace('p', '.')) if m else None
        short_mean = np.mean(rd['short']) if rd['short'] else None
        long_mean = np.mean(rd['long']) if rd['long'] else None
        gap = (long_mean - short_mean) if (short_mean is not None and long_mean is not None) else None
        rows6.append({'run_id': run_id, 'pref_high': high,
                      'short_mean_len': short_mean, 'long_mean_len': long_mean,
                      'gap': gap,
                      'n_short': len(rd['short']), 'n_long': len(rd['long'])})
    df6r = pd.DataFrame(rows6).dropna(subset=['pref_high']).sort_values('pref_high')

    if not df6r.empty and 'gap' in df6r.columns:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].plot(df6r['pref_high'], df6r['gap'], 'o-', color='tab:blue')
        axes[0].set_xlabel('pref_strength high');
        axes[0].set_ylabel('Mean route length gap (long - short)')
        axes[0].set_title('Preference Gap vs Strength')
        axes[1].plot(df6r['pref_high'], df6r['short_mean_len'], 'o-', label='short pref')
        axes[1].plot(df6r['pref_high'], df6r['long_mean_len'], 's-', label='long pref')
        axes[1].set_xlabel('pref_strength high')
        axes[1].set_title('Avg accepted route length by type')
        axes[1].legend(fontsize=9)
        plt.tight_layout();
        plt.show()
        display(df6r[['pref_high', 'short_mean_len', 'long_mean_len', 'gap', 'n_short', 'n_long']].reset_index(
            drop=True).round(2))

    print('\nRecommendation')
    if not df6r.empty and df6r['gap'].notna().any():
        try:
            avg_req_len = base_conf_sat.get('avg_request_lengths', 50)
        except NameError:
            avg_req_len = 50
        min_gap = avg_req_len * 0.3
        good6 = df6r[df6r['gap'] >= min_gap]
        if not good6.empty:
            rec_high = float(good6['pref_high'].min())
            gap_val = float(good6[good6['pref_high'] == rec_high]['gap'].values[0])
            print(f'  ok  route_pref_strength_range: low={ROUTE_PREF_LOW}, high={rec_high}')
            print(f'  Gap at this strength: {gap_val:.1f} units (target >= {min_gap:.1f})')
        else:
            rec_high = float(df6r['pref_high'].max())
            print(f'  ~ largest tested high ({rec_high}) gives gap {df6r["gap"].max():.1f};')
            print(f'    consider adding higher values to ROUTE_PREF_HIGH_LIST')
    else:
        rec_high = ROUTE_PREF_HIGH_LIST[-1]
        print(f'  ? no gap data - check taxi_static files exist in {RESULTS_ROUTE}')

    print(f'  nonpreferred_accept_ceiling     : {ROUTE_NONPREF_CEILING}')
    print(f'  preference_base_acceptance_prob : {ROUTE_BASE_ACCEPT_PROB}')
    print()
    print('  Add to base config:')
    print(f'    "route_pref_strength_range": {{"low": {ROUTE_PREF_LOW}, "high": {rec_high}}}')
    print(f'    "nonpreferred_accept_ceiling": {ROUTE_NONPREF_CEILING}')

    CAL['route_pref_strength_high'] = rec_high
    CAL['route_pref_strength_low'] = ROUTE_PREF_LOW

## Region preference

For `nearest_region_pref` and `nearest_two_sided_region_pass_pref`. Sweeps
`max_declines` against a `nearest` baseline. The popularity map in the regions
file should leave popular regions at the baseline service rate, let unpopular
ones drop 15-25 pp, and keep the overall drop below 10 pp.


In [ ]:
REG_BASE = CAL.get('base_config', BASE_CONF)
REG_D = CAL.get('d', 258)
REG_R = CAL.get('R', 0.5)
REG_GEOM = GEOM
REG_BEHAV = 1
REG_DAYS = 0.25
REG_DIR = f'configs/calibration/layer7_{GEOM_VARIANT}'

# how many declines a driver may make before forced acceptance
REG_MAX_DECLINES_LIST = [1, 2, 3, 5]

In [ ]:
regions_path = Path('configs') / REGIONS_FILE
if not regions_path.exists():
    print(f'Regions file not found: {regions_path}')
    print('Create it first, then re-run this cell.')
else:
    gen7 = ConfigGenerator(REG_BASE, days=REG_DAYS)
    configs7 = []

    conf_base = gen7.generate_config(REG_D, REG_R, 'nearest', REG_GEOM, REG_BEHAV,
                                     no_breaks=True, constant_rate=True)
    if conf_base:
        fname, content = gen7.dump_config(conf_base)
        configs7.append((fname, content))

    for md in REG_MAX_DECLINES_LIST:
        conf_reg = gen7.generate_config(REG_D, REG_R, 'nearest_region_pref', REG_GEOM, REG_BEHAV,
                                        regions_file=REGIONS_FILE,
                                        no_breaks=True, constant_rate=True)
        if conf_reg:
            conf_reg['max_declines'] = md
            fname, content = gen7.dump_config(conf_reg)
            fname = fname.replace('.conf', f'_md_{md}.conf')
            conf_dict = json.loads(content)
            conf_dict['_md_tag'] = md
            configs7.append((fname, json.dumps(conf_dict, indent=4) + '\n'))

    written7 = write_configs(configs7, REG_DIR)
    print('Configs:', [p.name for p in written7])

In [ ]:
ok7, fail7 = run_batch_parallel(written7)

In [ ]:
RESULTS_REG = Path(f'results/calibration/layer7_{GEOM_VARIANT}')

df7 = load_aggregates(RESULTS_REG)
svc7 = load_service_stats(RESULTS_REG)

if df7.empty:
    print('No results - run the sweep first.')
else:
    df7['service_rate'] = df7['run_id'].map({k: v['service_rate'] for k, v in svc7.items()})
    df7['is_region'] = df7['matching'].str.contains('region', na=False)
    df7['max_declines'] = df7['run_id'].str.extract(r'_md_(\d+)')[0].astype(float)

    base_sr = df7[~df7['is_region']]['service_rate'].mean()

    rows7 = []
    for md_val in sorted(df7['max_declines'].dropna().unique()):
        sub = df7[(df7['is_region']) & (df7['max_declines'] == md_val)]
        reg_sr = sub['service_rate'].mean()
        if pd.notna(base_sr) and pd.notna(reg_sr):
            drop = base_sr - reg_sr
            rows7.append({'max_declines': int(md_val), 'baseline_sr': base_sr,
                          'region_sr': reg_sr, 'drop': drop, 'ok': drop <= 0.10})

    df7r = pd.DataFrame(rows7)

    if not df7r.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(df7r['max_declines'], df7r['region_sr'], 'o-', color='tab:blue', label='region_pref')
        ax.axhline(base_sr, color='red', linestyle='--', label=f'baseline nearest ({base_sr:.1%})')
        ax.axhline(base_sr - 0.10, color='orange', linestyle=':', label='max 10pp drop')
        ax.set_xlabel('max_declines');
        ax.set_ylabel('Service rate')
        ax.set_title(f'Service Rate vs max_declines  [R={REG_R}]')
        ax.legend(fontsize=9)
        plt.tight_layout();
        plt.show()
        display(df7r.round(4))

    print(f'\nRecommendation  (baseline service rate: {base_sr:.1%})')
    good7 = df7r[df7r['ok']] if not df7r.empty else pd.DataFrame()
    if not good7.empty:
        rec_md = int(good7['max_declines'].min())
        row = good7[good7['max_declines'] == rec_md].iloc[0]
        print(f'  ok  max_declines = {rec_md}')
        print(f'    Service rate: {row["region_sr"]:.1%}  (drop: {row["drop"]:.1%})')
    else:
        rec_md = int(df7r['max_declines'].max()) if not df7r.empty else REG_MAX_DECLINES_LIST[-1]
        print(f'  ~ no value keeps drop <= 10pp; best is {rec_md}')
        print('    Consider adjusting popularity weights in regions.json')

    reg_files = list(RESULTS_REG.glob('run_*nearest_region*_region_safety_averages.csv.gz'))
    if reg_files:
        print('\nPer-region safety averages (first matching file):')
        rdf = pd.read_csv(reg_files[0])
        display(rdf.tail(3))

    CAL['max_declines'] = rec_md
    print(f'\n  CAL["max_declines"] = {rec_md}')

## Passenger safety preferences

Sweeps `passenger_score_temperature` with safety thresholds derived from the
observed safety score distribution. Strict passengers should end up with safer
drivers than indifferent passengers by a clear margin.


In [ ]:
PAX_BASE = CAL.get('base_config', BASE_CONF)
PAX_D = CAL.get('d', 258)
PAX_R = CAL.get('R', 0.5)
PAX_GEOM = GEOM
PAX_BEHAV = 1
PAX_DAYS = 1
PAX_DIR = f'configs/calibration/layer8_{GEOM_VARIANT}'

PAX_TEMPERATURES = [0.05, 0.10, 0.20, 0.40]
PAX_ALGORITHMS = ['nearest', 'nearest_passenger_pref']

# derive thresholds from the observed safety distribution; hardcoded values over-reject
# because scores degrade over the day (target: strict ~20-25% rejected, moderate ~10-15%)
_prev_results = Path(f'results/calibration/layer8_{GEOM_VARIANT}')
_safety_sample = []
for _f in sorted(_prev_results.glob('run_*_per_taxi_metrics.json.gz')):
    with gzip.open(_f, 'rt') as _fh:
        _lines = [_l for _l in _fh if _l.strip()]
    if _lines:
        _last = json.loads(_lines[-1])
        _safety_sample.extend([float(_s) for _s in _last.get('safety_score', []) if _s is not None])
    if len(_safety_sample) >= 200:
        break

if not _safety_sample:
    _safety3 = load_safety_scores(Path(f'results/calibration/layer3_{GEOM_VARIANT}'))
    for _sc in _safety3.values():
        _safety_sample.extend(_sc.get('final', []))

if _safety_sample:
    _p10 = float(np.percentile(_safety_sample, 10))
    _p25 = float(np.percentile(_safety_sample, 25))
    _std = float(np.std(_safety_sample))
    _src_label = 'own equilibrium' if (
            _prev_results.exists() and any(_prev_results.glob('*.gz'))
    ) else 'safety-decay proxy (re-run this cell once passenger runs exist)'
    print(f'Safety distribution ({_src_label}):')
    print(f'  mean={np.mean(_safety_sample):.1f}  p10={_p10:.1f}  p25={_p25:.1f}  std={_std:.1f}')
    PAX_SAFETY_THRESHOLDS = {
        'safety_indifferent': {'mean': 0.0, 'std': 0.0},
        'safety_moderate': {'mean': round(_p10, 1), 'std': round(_std * 0.3, 1)},
        'safety_strict': {'mean': round(_p25, 1), 'std': round(_std * 0.3, 1)},
    }
    print(f'  -> moderate threshold: {PAX_SAFETY_THRESHOLDS["safety_moderate"]["mean"]}  (~10% rejected)')
    print(f'  -> strict  threshold: {PAX_SAFETY_THRESHOLDS["safety_strict"]["mean"]}  (~25% rejected)')
else:
    print('WARNING: no safety data; using conservative fallback thresholds.')
    PAX_SAFETY_THRESHOLDS = {
        'safety_indifferent': {'mean': 0.0, 'std': 0.0},
        'safety_moderate': {'mean': 55.0, 'std': 8.0},
        'safety_strict': {'mean': 65.0, 'std': 6.0},
    }

In [ ]:
gen8 = ConfigGenerator(PAX_BASE, days=PAX_DAYS)
configs8 = []
for alg in PAX_ALGORITHMS:
    for T in PAX_TEMPERATURES:
        conf = gen8.generate_config(PAX_D, PAX_R, alg, PAX_GEOM, PAX_BEHAV,
                                    no_breaks=False, constant_rate=False)
        if conf is None:
            continue
        conf['passenger_score_temperature'] = T
        conf['passenger_safety_threshold'] = PAX_SAFETY_THRESHOLDS
        for key in ['income_target_rate', 'satisfaction_income_ref',
                    'satisfaction_change_waiting_rate', 'safety_score_change_serving_rate',
                    'safety_score_break_recovery_constant', 'driver_flexibility_threshold',
                    'driver_flexibility_temperature']:
            if key in CAL and CAL[key] is not None:
                conf[key] = CAL[key]
        T_tag = str(T).replace('.', 'p')
        fname, content = gen8.dump_config(conf)
        fname = fname.replace('.conf', f'_ptemp_{T_tag}.conf')
        conf_dict = json.loads(content)
        conf_dict['_ptemp_tag'] = T
        configs8.append((fname, json.dumps(conf_dict, indent=4) + '\n'))

written8 = write_configs(configs8, PAX_DIR)
print('Configs:', [p.name for p in written8])

In [ ]:
ok8, fail8 = run_batch_parallel(written8)

In [ ]:
RESULTS_PAX = Path(f'results/calibration/layer8_{GEOM_VARIANT}')


def load_passenger_acceptance(results_dir):
    # gap metric: fraction of completed rides where the driver's average safety score
    # is below the passenger's threshold - captures whether strict passengers
    # actually end up with safer drivers
    out = {}
    for f in sorted(Path(results_dir).glob('run_*_per_request_metrics.json.gz')):
        run_id = re.sub(r'^run_', '', re.sub(r'_per_request_metrics\.json\.gz$', '', f.name))
        accepted = {}
        rejected = {}
        below_thresh = {}
        total_done = {}
        with gzip.open(f, 'rt') as fh:
            for line in fh:
                if line.strip():
                    for req in json.loads(line).get('requests', []):
                        ptype = req.get('passenger_type', 'unknown')
                        if req.get('mode') == 'done':
                            accepted[ptype] = accepted.get(ptype, 0) + 1
                            total_done[ptype] = total_done.get(ptype, 0) + 1
                            ss = req.get('driver_average_safety_score')
                            thresh = req.get('safety_threshold', 0)
                            if ss is not None and thresh is not None and float(ss) < float(thresh):
                                below_thresh[ptype] = below_thresh.get(ptype, 0) + 1
                        elif req.get('mode') == 'dropped':
                            reason = req.get('cancellation_reason', '')
                            if 'passenger' in str(reason).lower() or 'pref' in str(reason).lower():
                                rejected[ptype] = rejected.get(ptype, 0) + 1
        if accepted or rejected:
            all_types = set(list(accepted.keys()) + list(rejected.keys()))
            rate = {}
            bt_rate = {}
            for pt in all_types:
                a = accepted.get(pt, 0);
                r = rejected.get(pt, 0)
                rate[pt] = a / (a + r) if (a + r) > 0 else None
                n = total_done.get(pt, 0)
                bt_rate[pt] = below_thresh.get(pt, 0) / n if n > 0 else None
            out[run_id] = {'accept_rate': rate, 'below_thresh_rate': bt_rate}
    return out


pax8 = load_passenger_acceptance(RESULTS_PAX)
svc8 = load_service_stats(RESULTS_PAX)
df8 = load_aggregates(RESULTS_PAX)
safety8 = load_safety_scores(RESULTS_PAX)

if df8.empty:
    print('No results - run the sweep first.')
else:
    df8['service_rate'] = df8['run_id'].map({k: v['service_rate'] for k, v in svc8.items()})
    df8 = add_utilization(df8)

    m = df8['run_id'].str.extract(r'_ptemp_([\dp]+)')
    df8['temperature'] = m[0].str.replace('p', '.').astype(float)

    rows8 = []
    for run_id, data in pax8.items():
        m2 = re.search(r'_ptemp_([\dp]+)', run_id)
        T = float(m2.group(1).replace('p', '.')) if m2 else None
        alg = parse_run_id(run_id).get('matching', '?')
        row = {'run_id': run_id, 'temperature': T, 'matching': alg}
        row.update({f'accept_{k}': v for k, v in data['accept_rate'].items()})
        row.update({f'bt_{k}': v for k, v in data['below_thresh_rate'].items()})
        rows8.append(row)
    df8r = pd.DataFrame(rows8)

    pax_types = ['safety_indifferent', 'safety_moderate', 'safety_strict']

    pref_df = df8r[df8r['matching'].str.contains('pref', na=False)].copy()
    bt_cols_present = [f'bt_{pt}' for pt in pax_types if f'bt_{pt}' in pref_df.columns]
    if not pref_df.empty and bt_cols_present:
        fig, ax = plt.subplots(figsize=(10, 4))
        temps = sorted(pref_df['temperature'].dropna().unique())
        colors = ['tab:blue', 'tab:orange', 'tab:red']
        for pt, col in zip(pax_types, colors):
            bt_col = f'bt_{pt}'
            if bt_col in pref_df.columns:
                vals = [pref_df[pref_df['temperature'] == T][bt_col].mean() for T in temps]
                ax.plot(temps, vals, 'o-', label=pt, color=col)
        ax.axhspan(0, 0.10, alpha=0.12, color='green', label='target: <10% below-threshold')
        ax.set_xlabel('passenger_score_temperature')
        ax.set_ylabel('Fraction of rides with driver below\npassenger safety threshold')
        ax.set_title('Passenger preference differentiation vs temperature\n'
                     '(lower = better match quality for strict/moderate passengers)')
        ax.legend(fontsize=9)
        plt.tight_layout();
        plt.show()

    safety_vals = []
    for rid, sc in safety8.items():
        safety_vals.extend(sc.get('final', []))
    if safety_vals:
        fig2, ax2 = plt.subplots(figsize=(8, 3))
        ax2.hist(safety_vals, bins=30, color='tab:green', edgecolor='white', alpha=0.8)
        for pt, threshold in [('strict', PAX_SAFETY_THRESHOLDS.get('safety_strict', {}).get('mean', 75)),
                              ('moderate', PAX_SAFETY_THRESHOLDS.get('safety_moderate', {}).get('mean', 55))]:
            frac_below = np.mean([s < threshold for s in safety_vals])
            ax2.axvline(threshold, linestyle='--',
                        label=f'{pt} threshold={threshold:.1f} ({frac_below:.0%} drivers below)')
        ax2.set_xlabel('Safety score');
        ax2.set_title('Safety score distribution (end of run)')
        ax2.legend(fontsize=9)
        plt.tight_layout();
        plt.show()

    print('\nRecommendation')
    if not pref_df.empty and 'bt_safety_strict' in pref_df.columns and 'bt_safety_indifferent' in pref_df.columns:
        by_temp = pref_df.groupby('temperature')[['bt_safety_indifferent', 'bt_safety_strict']].mean()
        by_temp['gap'] = by_temp['bt_safety_strict'] - by_temp['bt_safety_indifferent']
        print(f'  below-threshold rates by temperature (pref algorithm):')
        for T, row in by_temp.iterrows():
            marker = 'ok' if row['gap'] > 0.15 else '~'
            print(f'    {marker}  T={T}: indifferent={row["bt_safety_indifferent"]:.1%}, '
                  f'strict={row["bt_safety_strict"]:.1%}, gap={row["gap"]:.1%}')
        good8 = by_temp[by_temp['gap'] > 0.15]
        if not good8.empty:
            # prefer the sharpest T that still shows a clear gap
            rec_T = float(good8.index.min())
            rec_row = good8.loc[rec_T]
            print(f'\n  ok  passenger_score_temperature = {rec_T}')
            print(f'    Strict below-threshold rate: {rec_row["bt_safety_strict"]:.1%}  '
                  f'(gap vs indifferent: {rec_row["gap"]:.1%})')
        else:
            rec_T = float(PAX_TEMPERATURES[0])
            print(f'\n  ~ gap < 15pp for all tested temperatures; using sharpest T={rec_T}')
            print(f'    Consider raising strict threshold or verifying algorithm safety-score weighting.')
    else:
        rec_T = 0.10
        print(f'  ? below-threshold data not available - defaulting to T={rec_T}')

    if safety_vals:
        strict_thresh = PAX_SAFETY_THRESHOLDS.get('safety_strict', {}).get('mean', 75)
        frac_rejected = np.mean([s < strict_thresh for s in safety_vals])
        print(f'\n  {frac_rejected:.0%} of drivers end below strict threshold ({strict_thresh:.1f})')
        if frac_rejected < 0.05:
            print('  -> too few drivers below threshold: lower strict threshold or decrease safety_score_min')
        elif frac_rejected > 0.50:
            print('  -> too many drivers below threshold: raise threshold or adjust the safety decay calibration')
        else:
            print('  -> fraction in good range (5-50%)')

    print()
    print('  Add to base config:')
    print(f'    "passenger_score_temperature": {rec_T}')
    print(f'    "passenger_safety_threshold": (from PAX_SAFETY_THRESHOLDS above)')

    CAL['passenger_score_temperature'] = rec_T
    print(f'\n  CAL["passenger_score_temperature"] = {rec_T}')


## Calibrated values


In [ ]:
print('-' * 50)
print('  CALIBRATION SUMMARY')
print('-' * 50)
sections = [
    ('supply/demand', ['R', 'd']),
    ('income', ['income_target_rate', 'mean_income_rate', 'median_trip_income']),
    ('safety', ['safety_score_change_serving_rate', 'safety_score_break_recovery_constant']),
    ('satisfaction', ['satisfaction_income_ref', 'satisfaction_change_waiting_rate']),
    ('flexibility', ['driver_flexibility_threshold', 'driver_flexibility_temperature']),
    ('route preference', ['route_pref_strength_low', 'route_pref_strength_high']),
    ('passenger preference', ['passenger_score_temperature']),
]
for section, keys in sections:
    print(f'\n  {section}')
    for k in keys:
        v = CAL.get(k)
        if v is not None:
            print(f'    {k:<45s} = {v}')
        else:
            print(f'    {k:<45s}   (not set - run that section)')
print()
print('-' * 50)

In [ ]:
EXPORT_PATH = Path('configs/calibration_patch.json')

EXPORTABLE = [
    'income_target_rate',
    'safety_score_change_serving_rate',
    'safety_score_break_recovery_constant',
    'satisfaction_income_ref',
    'satisfaction_change_waiting_rate',
    'driver_flexibility_threshold',
    'driver_flexibility_temperature',
    'passenger_score_temperature',
]
patch = {k: CAL[k] for k in EXPORTABLE if k in CAL and CAL[k] is not None}

if 'route_pref_strength_low' in CAL and 'route_pref_strength_high' in CAL:
    patch['route_pref_strength_range'] = {
        'low': CAL['route_pref_strength_low'],
        'high': CAL['route_pref_strength_high'],
    }

EXPORT_PATH.write_text(json.dumps(patch, indent=4) + '\n')
print(f'Patch written to {EXPORT_PATH}:')
print(json.dumps(patch, indent=4))